In [ ]:
import random
from pathlib import Path
from collections import defaultdict
import torch

from sklearn.model_selection import GroupKFold, KFold

from monai.data import CacheDataset, DataLoader
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Resized,
    AsDiscreted,
    ToTensord,

    RandSpatialCropd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandAdjustContrastd,
    RandShiftIntensityd,
    Compose,
    MapTransform
)

In [ ]:
import os
print(os.getcwd())

In [ ]:
import torch
import random
from monai.transforms import MapTransform

class TemporalPriorLogicd(MapTransform):
    """
    Trasformazione custom per MONAI.
    Gestisce il caso t=0 azzerando la maschera precedente e applica 
    il Mask Dropout per forzare il modello a guardare i canali RGB.
    """
    def __init__(self, keys, prev_mask_key="prev_label", dropout_prob=0.2):
        super().__init__(keys)
        self.prev_mask_key = prev_mask_key
        self.dropout_prob = dropout_prob

    def __call__(self, data):
        d = dict(data)
        
        # Se 'is_first_frame' è True, la prev_label deve essere un tensore di zeri
        if d.get('is_first_frame', False):
            d[self.prev_mask_key] = torch.zeros_like(d[self.prev_mask_key])
            
        # Con una probabilità 'dropout_prob' (es. 20%), azzeriamo la maschera 
        # per forzare la rete a non dipendere solo dal prior spaziale
        elif random.random() < self.dropout_prob:
            d[self.prev_mask_key] = torch.zeros_like(d[self.prev_mask_key])
            
        return d

In [ ]:
class HemosetDataSet:
    def __init__(self, root_dir="../data/raw", image_size=(640, 480), seed=42):
        self.root_dir = Path(root_dir)
        self.image_size = image_size

        self.rng = random.Random(seed)

        if not self.root_dir.exists():
            raise FileNotFoundError(f"La directory {self.root_dir} non esiste.")

        self.patient_data = defaultdict(list)

        image_paths = sorted(list(self.root_dir.rglob("*/imgs/**/*.png")))

        for img_path in image_paths:
            # img_path.relative_to(root_dir) diventa "pig1/imgs/imgs/000000.png"
            # .parts[0] estrae esattamente "pig1"
            patient_id = img_path.relative_to(self.root_dir).parts[0]
            
            frame_name = img_path.stem 

            mask_path_png = self.root_dir / patient_id / "labels" / "labels" /f"{frame_name}_mask.png"

            if mask_path_png.exists():
                final_mask_path = mask_path_png
            
            else:
                print(f"[Warning] Maschera mancante per l'immagine {img_path.name}. Skip.")
                continue

            self.patient_data[patient_id].append({
                "image": str(img_path),
                "label": str(final_mask_path)
            })

        if not self.patient_data:
            raise RuntimeError("Nessun dato accoppiato (img/mask) trovato. Verifica la struttura delle cartelle.")

        print(f"[*] Dataset caricato: trovati {len(self.patient_data)} subjects (pigN) distinti.")
        print(f"[*] Totale frame validi: {sum(len(frames) for frames in self.patient_data.values())}")

        self.base_transforms = Compose([
            LoadImaged(keys=["image", "label", "prev_label"], reader="PILReader"),
            EnsureChannelFirstd(keys=["image", "label", "prev_label"]),
            ScaleIntensityRanged(keys=["image"], a_min=0, a_max=255, b_min=0.0, b_max=1.0, clip=True),            
            AsDiscreted(keys=["label", "prev_label"], threshold=0.5),
            Resized(keys=["image", "label", "prev_label"], spatial_size=self.image_size, mode=("bilinear", "nearest", "nearest")),
            ToTensord(keys=["image", "label", "prev_label"], dtype=torch.float32),
        ])

    def get_loaders(self, fold_idx=0, n_splits=5, cache_rate=1.0, batch_size=4, num_workers=4, train_transforms=None):
        
        patients = sorted(list(self.patient_data.keys()))
        self.rng.shuffle(patients)

        if n_splits > len(patients):
            raise ValueError("n_splits > numero di pig")

        patients = patients.copy()
        self.rng.shuffle(patients)

        kf = KFold(n_splits=n_splits, shuffle=False)
        folds = list(kf.split(patients))

        if fold_idx >= len(folds):
            raise ValueError(f"fold_idx deve essere < {n_splits}")

        train_val_idx, test_idx = folds[fold_idx]

        train_val_patients = [patients[i] for i in train_val_idx]
        test_patients = [patients[i] for i in test_idx]

        val_size = max(1, int(0.25 * len(train_val_patients)))
        val_patients = train_val_patients[-val_size:]
        train_patients = train_val_patients[:-val_size]

        print("\n[*] Fold info")
        print(f"Train pigs: {train_patients}")
        print(f"Val pigs:   {val_patients}")
        print(f"Test pigs:  {test_patients}")

        def build_temporal_files(patient_id):
            """Genera dicts contenenti frame_t, label_t e prev_label_t-1"""
            files = self.patient_data[patient_id] # Assunti già ordinati per timestamp
            temp_list = []
            for i, item in enumerate(files):
                new_item = item.copy()
                if i == 0:
                    # Per t=0 non esiste un t-1. Usiamo un flag per gestirlo nelle transforms.
                    # Assegniamo temporaneamente la label corrente a prev_label per non 
                    # rompere le transform di LoadImaged, poi la azzereremo.
                    new_item['prev_label'] = item['label']
                    new_item['is_first_frame'] = True
                else:
                    new_item['prev_label'] = files[i-1]['label']
                    new_item['is_first_frame'] = False
                temp_list.append(new_item)
            return temp_list

        train_files = []
        for p in train_patients:
            train_files.extend(build_temporal_files(p))

        val_files = []
        for p in val_patients:
            val_files.extend(build_temporal_files(p))

        test_files = []
        for p in test_patients:
            test_files.extend(build_temporal_files(p))

        self.rng.shuffle(train_files)

        print(
            f"[*] Samples "
            f"train={len(train_files)} "
            f"val={len(val_files)} "
            f"test={len(test_files)}"
        )

        train_compose = (Compose([self.base_transforms, train_transforms]) if train_transforms else self.base_transforms)

        train_ds = CacheDataset(train_files, transform=train_compose, cache_rate=cache_rate)
        val_ds = CacheDataset(val_files, transform=self.base_transforms, cache_rate=cache_rate)
        test_ds = CacheDataset(test_files, transform=self.base_transforms, cache_rate=cache_rate)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    
        return train_loader, val_loader, test_loader

    def get_sample(self, split="train", index=None, patient_id=None, transform=True):

        patients = sorted(list(self.patient_data.keys()))
        temp_rng = random.Random(42) 
        temp_rng.shuffle(patients)

        split_idx = int(0.8 * len(patients))
        train_patients = patients[:split_idx]
        val_patients = patients[split_idx:]

        selected_patients = train_patients if split == "train" else val_patients

        if patient_id:
            if patient_id not in selected_patients:
                raise ValueError(f"{patient_id} non è nello split {split}")
            files = self.patient_data[patient_id]
        else:
            files = []
            for p in selected_patients:
                files.extend(self.patient_data[p])

        if index is None:
            sample = random.choice(files)
        else:
            sample = files[index % len(files)]

        if transform:
            sample = self.base_transforms(sample)

        return sample

In [ ]:
hemoset = HemosetDataSet()

In [ ]:
train_trasformer = Compose([
    RandSpatialCropd(keys=['image', 'label', 'prev_label'], roi_size=(320, 320), random_size=False), 
    RandAdjustContrastd(keys=["image"], prob=0.5, gamma=(0.5, 1.5)),
    TemporalPriorLogicd(keys=["prev_label"], dropout_prob=0.2)
])

train_loader, val_loader, test_loader = hemoset.get_loaders(train_transforms=train_trasformer)

In [ ]:
iterator = iter(train_loader)

In [ ]:
batch = next(iterator)

In [ ]:
len(batch)

In [ ]:
batch["image"].shape, batch["prev_label"].shape, batch["label"].shape

In [ ]:
from monai.networks.nets import UNet

In [ ]:
unet = UNet(
    spatial_dims = 2,
    in_channels  = 3,
    out_channels = 1,
    channels = [32, 64, 128, 256, 512],
    strides = [2, 2, 2, 2],
    num_res_units = 2
).cuda()

In [ ]:
unet

In [ ]:
conv0= unet.model[0].conv.unit0.conv

In [ ]:
residual_conv0 = unet.model[0].residual

In [ ]:
conv0

In [ ]:
residual_conv0

In [ ]:
def expand_conv_channels(old_conv):
    new_conv0= torch.nn.Conv2d(
        in_channels=4,
         out_channels=conv0.out_channels,
        kernel_size=conv0.kernel_size,
        stride=conv0.stride,
        padding=conv0.padding,
        bias=conv0.bias is not None
    ).to(conv0.weight.device)

    # copia pesi RGB
    with torch.inference_mode():
        new_conv0.weight[:, :3] = conv0.weight
    
        # inizializza il nuovo canale
        new_conv0.weight[:, 3:] = conv0.weight.mean(dim=1, keepdim=True)
    
        if conv0.bias is not None:
            new_conv0.bias.copy_(conv0.bias)
    return new_conv0

unet.model[0].conv.unit0.conv = expand_conv_channels(conv0)
unet.model[0].residual = expand_conv_channels(residual_conv0)
print(f"[*] Canali in ingresso aggiornati a: {unet.model[0].conv.unit0.conv.in_channels}")

In [ ]:
unet = unet.cuda()

In [ ]:
import torch
from monai.losses import DiceCELoss
from tqdm import tqdm
from monai.metrics import DiceMetric

# Inizializziamo la metrica di MONAI per la validazione
dice_metric = DiceMetric(include_background=False, reduction="mean")

max_epochs = 10

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)

scaler = torch.cuda.amp.GradScaler(enabled=True)

criterion = DiceCELoss(sigmoid=True, lambda_dice=0.5, lambda_ce=0.5)

early_stopping = {
    "patience": 5,
    "monitor": "val_dice",
    "mode": "max"
}

In [ ]:
for epoch in range(max_epochs):

    #train step
    unet.train()
    epoch_loss = 0.0
    train_pbar= tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs} [Train]")
    
    for batch in train_loader:
        image= batch["image"].cuda()
        mask= batch["label"].cuda() 
        maskt= batch["prev_label"].cuda() 

        input = torch.cat((image, maskt), dim=1)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            pred = unet(input)
            loss = criterion(pred, mask)
            
        #backwarpass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_train_loss = epoch_loss / len(train_loader)
    scheduler.step()
    
    #eval step
    unet.eval()
    pred_mask_prev = None

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{max_epochs} [Val]")
    
    with torch.inference_mode():
         for batch in train_loader:
            image= batch["image"].cuda()
            B, _, H, W = image.shape
            mask_gt = batch["label"].cuda()
             
            if batch.get('is_first_frame', [False])[0] or pred_mask_prev is None:
                pred_mask_prev = torch.zeros((B, 1, H, W)).cuda()
            
            inputs_4c = torch.cat([image, pred_mask_prev], dim=1)
            pred=unet(input)
    
            prob_mask = torch.sigmoid(pred)
            current_pred_binary = (prob_mask > 0.5).float()
                
            pred_mask_prev = current_pred_binary
            dice_metric(y_pred=current_pred_binary, y=mask_gt)
         val_dice = dice_metric.aggregate().item()
         dice_metric.reset()
    print(f"\n=> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Dice: {val_dice:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}\n")
            